# (7065) Fredschaaf — просмотр и красивые картинки

Это лёгкий notebook для знакомства с наблюдениями: выбрать серию, открыть кадр, посмотреть WCS-сетку, сравнить несколько экспозиций, сделать вырезку и простой стек.

Он не делает точную астрометрию и не пытается автоматически найти астероид. Сложный экспериментальный конвейер сохранён отдельно в `reductions_advanced.ipynb`.

Запускайте из корня проекта через `./run_jupyter.sh`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from astropy.io import fits
from astropy.visualization import AsinhStretch, ImageNormalize
from astropy.wcs import WCS

plt.rcParams.update({
    "figure.figsize": (11, 7),
    "image.cmap": "magma",
    "axes.grid": False,
})

## 1. Выберите серию

Начните с серии `20250903` в фильтре `R`: у неё есть WCS, поэтому на изображении можно показать координатную сетку. Для H-alpha поменяйте `FILTER`, для другой ночи — `DATE`.

In [ ]:
ROOT = Path.cwd()
DATA_ROOT = ROOT / "Fredschaaf"

DATE = "20250903"
FILTER = "R"

paths = sorted(path for path in (DATA_ROOT / DATE).glob("*.fits") if f"_{FILTER}_" in path.name)
assert paths, f"Не нашлись кадры {DATE} / {FILTER} в {DATA_ROOT}"

print(f"Найдено кадров: {len(paths)}")
for number, path in enumerate(paths[:10]):
    print(f"{number:>2}: {path.name}")
if len(paths) > 10:
    print("…")

## 2. Один кадр

Поменяйте `FRAME_NUMBER`, чтобы посмотреть другую экспозицию. Сетка появляется только там, где в FITS уже есть WCS. Масштаб яркости подбирается для каждого кадра автоматически — слабые звёзды лучше видны, а яркие не превращают всё поле в белый прямоугольник.

In [ ]:
def read_frame(path):
    """Прочитать один FITS-кадр и сразу закрыть файл."""
    with fits.open(path, memmap=False) as hdul:
        return hdul[0].data.astype(np.float32), hdul[0].header.copy()


def make_norm(data):
    """Мягкая шкала яркости для просмотра слабых объектов."""
    finite = data[np.isfinite(data)]
    vmin, vmax = np.percentile(finite, (5, 99.7))
    return ImageNormalize(vmin=vmin, vmax=vmax, stretch=AsinhStretch(a=0.08))


def show_frame(path, *, with_wcs=True, title=None, ax=None):
    """Нарисовать кадр; при наличии WCS добавить координатную сетку."""
    data, header = read_frame(path)
    wcs = WCS(header)
    has_wcs = with_wcs and wcs.has_celestial

    if ax is None:
        fig = plt.figure(figsize=(12, 8))
        ax = fig.add_subplot(projection=wcs) if has_wcs else fig.add_subplot()
    image = ax.imshow(data, origin="lower", norm=make_norm(data))

    if has_wcs:
        ax.coords.grid(color="white", linestyle=":", alpha=0.55)
        ax.set_xlabel("Прямое восхождение")
        ax.set_ylabel("Склонение")
    else:
        ax.set_xlabel("x, пиксели")
        ax.set_ylabel("y, пиксели")

    ax.set_title(title or path.name)
    return data, header, image


FRAME_NUMBER = 0
frame_data, frame_header, image = show_frame(paths[FRAME_NUMBER])
plt.colorbar(image, label="счёт, ADU")
plt.show()

## 3. Небольшая галерея

Удобна, чтобы быстро увидеть качество ведения, облака или изменения фона. Здесь кадры открываются по одному: серия целиком в память не загружается.

In [ ]:
GALLERY_NUMBERS = [0, 10, 30, 60]
GALLERY_NUMBERS = [number for number in GALLERY_NUMBERS if number < len(paths)]

fig, axes = plt.subplots(1, len(GALLERY_NUMBERS), figsize=(5 * len(GALLERY_NUMBERS), 5), constrained_layout=True)
axes = np.atleast_1d(axes)
for number, ax in zip(GALLERY_NUMBERS, axes):
    data, _ = read_frame(paths[number])
    ax.imshow(data, origin="lower", norm=make_norm(data))
    ax.set_title(f"№ {number}")
    ax.set_axis_off()
plt.show()

## 4. Вырезка интересной области

Сначала выберите координаты по курсору на полном изображении (Matplotlib показывает их внизу окна), затем впишите их в `CENTER_X`, `CENTER_Y`. Это особенно удобно для астероида, яркой звезды или дефекта кадра.

In [ ]:
CENTER_X, CENTER_Y = 1000, 700  # поменяйте на интересующую точку
HALF_SIZE = 80                   # половина стороны вырезки, пиксели


def show_cutout(data, x, y, half_size=80):
    x0, x1 = int(x - half_size), int(x + half_size)
    y0, y1 = int(y - half_size), int(y + half_size)
    cutout = data[y0:y1, x0:x1]

    fig, ax = plt.subplots(figsize=(7, 7))
    image = ax.imshow(cutout, origin="lower", norm=make_norm(cutout),
                      extent=(x0, x1, y0, y1))
    ax.scatter([x], [y], s=120, facecolors="none", edgecolors="cyan", linewidths=1.8)
    ax.set_xlabel("x, пиксели")
    ax.set_ylabel("y, пиксели")
    ax.set_title(f"Вырезка вокруг x={x:.0f}, y={y:.0f}")
    plt.colorbar(image, ax=ax, label="счёт, ADU")
    plt.show()


show_cutout(frame_data, CENTER_X, CENTER_Y, HALF_SIZE)

## 5. Простой стек для красивого изображения

Стек строится только из первых `STACK_COUNT` кадров и обрабатывает их по одному, поэтому не требует держать всю серию в памяти. Это **не** точный научный стек: кадры здесь не выравниваются. Он полезен для наглядной картинки и первого знакомства с полем.

Увеличивайте `STACK_COUNT` постепенно и запускайте ячейку только тогда, когда захотите построить стек.

In [ ]:
STACK_COUNT = 12


def simple_mean_stack(selected_paths):
    """Сложить кадры последовательно, не загружая весь набор в память."""
    total = None
    for path in selected_paths:
        data, _ = read_frame(path)
        background = np.nanmedian(data)
        if total is None:
            total = np.zeros_like(data, dtype=np.float32)
        total += data - background
    return total / len(selected_paths)


selected_paths = paths[:STACK_COUNT]
stack = simple_mean_stack(selected_paths)

fig, ax = plt.subplots(figsize=(12, 8))
image = ax.imshow(stack, origin="lower", norm=make_norm(stack))
ax.set_title(f"Простой стек: {len(selected_paths)} кадров, {DATE}, {FILTER}")
ax.set_xlabel("x, пиксели")
ax.set_ylabel("y, пиксели")
plt.colorbar(image, ax=ax, label="фон-вычтенный счёт, ADU")
plt.show()

## Что дальше

- Для красивой картинки меняйте номер кадра, область вырезки и число кадров в стеке.
- Не запускайте сложную обработку, пока не станет понятно, где находится объект и как выглядит серия.
- Когда понадобится выравнивание по звёздам, поиск астероида или измерение координат, откройте `reductions_advanced.ipynb`.